In [152]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [153]:
df = pd.read_csv('./data/spam_preprocessed.csv', encoding='latin-1')
# vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=3000, min_df=2)
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=3000, min_df=1)

In [154]:
x = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train_vectorized = vectorizer.fit_transform(X_train)
x_test_vectorized = vectorizer.transform(X_test)

In [155]:
print(f"x_train shape: {x_train_vectorized.shape}")
print(f"x_test shape: {x_test_vectorized.shape}")
classifier = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    C=1.0,
    class_weight='balanced',
    random_state=4239
)
model = classifier.fit(x_train_vectorized, y_train)
y_pred = model.predict(x_test_vectorized)

x_train shape: (4456, 3000)
x_test shape: (1114, 3000)


In [156]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9811
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       965
           1       0.93      0.93      0.93       149

    accuracy                           0.98      1114
   macro avg       0.96      0.96      0.96      1114
weighted avg       0.98      0.98      0.98      1114

Confusion Matrix:
[[954  11]
 [ 10 139]]


In [157]:
test_spam_data = pd.read_csv('./data/spam_test_preprocessed.csv')
# test_spam_data.replace({'label': {'ham': 0, 'spam': 1}}, inplace=True)
test_spam_data = test_spam_data[0:1]
test_spam_data['text'] = 'make this summer fashionable with monte carlo eoss deals! enjoy up to 30% off and upgrade your wardrobe today. visit your nearest showroom. t&c apply.'
test_spam_data['text'] = 'delivery attempt for sp_inland_doc no: ji297250619in is successful on 22-05-2026 15:49:56 - indiapost'
test_spam_data['label'] = 0

In [158]:
test_spam_data['vectorized_message'] = vectorizer.transform(test_spam_data['text']).toarray().tolist()
test_spam_data = test_spam_data.sample(frac=1, random_state=1290).reset_index(drop=True)
test_spam_data['label'] = test_spam_data['label'].astype(int)

In [159]:
y_test_labels = test_spam_data['label']
separate_X_test = np.vstack(test_spam_data['vectorized_message'])
random_y_pred = model.predict(separate_X_test)
confidence = model.predict_proba(separate_X_test)

In [160]:
test_accuracy = accuracy_score(y_test_labels, random_y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")
print("Test Classification Report:")
print(classification_report(y_test_labels, random_y_pred))
print("Test Confusion Matrix:")
print(confusion_matrix(y_test_labels, random_y_pred))

Test Accuracy: 0.0000
Test Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       1.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       1.0
   macro avg       0.00      0.00      0.00       1.0
weighted avg       0.00      0.00      0.00       1.0

Test Confusion Matrix:
[[0 1]
 [0 0]]


/Users/pygo/work/personal/courses/cba-gh/spam-detection/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pygo/work/personal/courses/cba-gh/spam-detection/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/pygo/work/personal/courses/cba-gh/spam-detection/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` param

In [161]:
test_spam_data['predicted'] = random_y_pred
test_spam_data['confidence_spam'] = confidence[:, 1] * 100
test_spam_data[['label', 'predicted', 'confidence_spam', 'text']]

,label,predicted,confidence_spam,text
0,0,1,56.460629,delivery attempt for sp_inland_doc no: ji29725...
